# ML 벤치마크
Purpose: compare CPU-only classical ML candidates using the immutable shared validation contract.

> Warning: this is an oracle/sanity-only synthetic-data benchmark, not real-device or medical-performance evidence.

In [ ]:
from __future__ import annotations

import hashlib
import json
from collections.abc import Sequence
from pathlib import Path
from typing import Any

import numpy as np
import pandas as pd
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    average_precision_score,
    brier_score_loss,
    f1_score,
    recall_score,
    roc_auc_score,
)
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

SERIES_ID = "mvp3-oracle-v1"
EXPECTED_SPLIT_COUNTS = {"train": 24, "validation": 6, "locked_test": 6}
DATA_STATUS = "oracle/sanity"
REAL_ACCURACY_STATUS = "NOT VERIFIED"
DEVICE_SYNCHRONIZATION_STATUS = "NOT_AVAILABLE_TRUTH_ONLY"
RUN_TRAINING = False
RUN_LOCKED_TEST = False

device = "cpu"
ML_VIEW_ROOT = Path("/kaggle/working/goal15_ml_view")
ML_BENCHMARK_OUTPUT_ROOT = Path("/kaggle/working/goal15_ml_benchmark")
STABLE_KEYS = ("person_key", "canonical_time")
PATTERN_TARGET = "pattern_binary"
ONSET_EVENT_TARGET = "event_binary"
STAGE_TARGET = "stage_code"
STAGE_CODES = ("LOW", "MEDIUM", "HIGH", "DECREASING", "RECOVERY")
BEHAVIOR_CODES = (
    "ear_covering",
    "exit_attempt",
    "head_turn_away",
    "motion_freeze",
    "movement_reduction",
    "repetitive_body_movement",
    "repetitive_hand_movement",
    "repetitive_object_contact",
    "sustained_pressure_or_contact",
    "withdrawal_movement",
)
REQUIRED_VIEW_COLUMNS = (
    *STABLE_KEYS,
    PATTERN_TARGET,
    ONSET_EVENT_TARGET,
    "hard_negative",
    STAGE_TARGET,
    *BEHAVIOR_CODES,
)

CAUSAL_FACTORS = (
    "autonomic_arousal",
    "motor_activation",
    "cognitive_load",
    "sleep_pressure",
    "sensory_context",
    "recovery_capacity",
    "social_context",
)
ROLLING_STATISTICS = ("mean", "std", "slope")
ROLLING_WINDOWS_SECONDS = (5, 15, 30, 60, 180, 300)
TIME_FEATURE_COLUMNS = (
    "time_sin",
    "time_cos",
    "weekday_sin",
    "weekday_cos",
    "is_awake",
)
CONTEXT_FEATURE_COLUMNS = (
    "context__sleep",
    "context__transition",
    "context__meal_context",
    "context__focused_task",
    "context__moderate_activity",
    "context__light_activity",
    "context__wake_rest",
    "context__sedentary_activity",
)
ALLOWED_FEATURE_COLUMNS = tuple(
    [
        feature
        for factor in CAUSAL_FACTORS
        for feature in (
            f"{factor}__robust_z",
            *(
                f"{factor}__{statistic}_{window_seconds}s"
                for window_seconds in ROLLING_WINDOWS_SECONDS
                for statistic in ROLLING_STATISTICS
            ),
        )
    ]
    + list(TIME_FEATURE_COLUMNS)
    + list(CONTEXT_FEATURE_COLUMNS)
)
EXACT_NON_FEATURE_COLUMNS = frozenset({
    "person_id",
    "person_key",
    "run_id",
    "dataset_id",
    "canonical_time",
    "timestamp_utc",
    "split_role",
    "context",
    "context_state",
    "event_id",
    "session_id",
    "day_key",
    "day",
    "date",
    "source_path",
    "label_source",
    "label_confidence",
    "standard_type",
    "ood_status",
    "event_type",
    "is_target",
    "hard_negative",
    "is_hard_negative",
    "hard_negative_kind",
    "event_label",
    "label",
    "missing_block_id",
    "forecast_60s",
    PATTERN_TARGET,
    ONSET_EVENT_TARGET,
    STAGE_TARGET,
    *BEHAVIOR_CODES,
})
PREDICTION_COLUMNS = [
    "model_family",
    "model_name",
    "series_id",
    "dataset_id",
    "run_id",
    "person_key",
    "day_key",
    "session_id",
    "canonical_time",
    "split_role",
    "target",
    "label",
    "probability",
    "threshold",
]
METRIC_COLUMNS = [
    "model_family",
    "model_name",
    "series_id",
    "split_role",
    "target",
    "metric",
    "value",
    "support",
    "data_status",
]


## 1. Validate ML role views
The benchmark fails closed unless each role view has the hash-verified shared identity and all event, stage, and behavior targets.

In [ ]:
def sha256_file(path: Path) -> str:
    digest = hashlib.sha256()
    with path.open("rb") as handle:
        for chunk in iter(lambda: handle.read(1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()


def _require_sha256(value: Any, field: str) -> str:
    if not isinstance(value, str) or len(value) != 64:
        raise ValueError(f"invalid SHA-256 for {field}")
    try:
        int(value, 16)
    except ValueError as exc:
        raise ValueError(f"invalid SHA-256 for {field}") from exc
    return value


def verify_ml_view_manifest(view_root: Path = ML_VIEW_ROOT) -> dict[str, Any]:
    manifest_path = view_root / "view_manifest.json"
    if not manifest_path.is_file():
        raise FileNotFoundError(f"missing ML view manifest: {manifest_path}")
    manifest = json.loads(manifest_path.read_text())
    if manifest.get("series_id") != SERIES_ID:
        raise ValueError("ML view manifest series_id mismatch")
    if manifest.get("data_status") != DATA_STATUS:
        raise ValueError("ML view manifest data_status mismatch")
    _require_sha256(manifest.get("source_dataset_hash"), "source_dataset_hash")
    _require_sha256(manifest.get("split_hash"), "split_hash")
    files = manifest.get("files")
    if not isinstance(files, dict) or set(files) != set(EXPECTED_SPLIT_COUNTS):
        raise ValueError("ML view manifest must declare exactly train, validation, locked_test")
    for split_role, metadata in files.items():
        if not isinstance(metadata, dict):
            raise ValueError(f"ML view metadata is invalid for {split_role}")
        path_name = metadata.get("path")
        if not isinstance(path_name, str) or Path(path_name).name != path_name:
            raise ValueError(f"ML view path is invalid for {split_role}")
        view_path = view_root / path_name
        if not view_path.is_file():
            raise FileNotFoundError(f"missing ML role view: {view_path}")
        if sha256_file(view_path) != _require_sha256(metadata.get("sha256"), path_name):
            raise ValueError(f"ML role view hash mismatch: {path_name}")
        if not isinstance(metadata.get("row_count"), int) or metadata["row_count"] < 1:
            raise ValueError(f"ML role view row_count is invalid for {split_role}")
        columns = metadata.get("columns")
        if not isinstance(columns, list) or not all(isinstance(column, str) for column in columns):
            raise ValueError(f"ML role view columns are invalid for {split_role}")
        missing = sorted(set(REQUIRED_VIEW_COLUMNS).difference(columns))
        if missing:
            raise ValueError(f"ML role view required columns missing for {split_role}: {missing}")
    return manifest


def _validate_loaded_split_contract(views: dict[str, pd.DataFrame]) -> None:
    person_sets = {
        split_role: set(frame["person_key"].astype(str))
        for split_role, frame in views.items()
    }
    counts = {split_role: len(people) for split_role, people in person_sets.items()}
    if counts != EXPECTED_SPLIT_COUNTS:
        raise ValueError(f"person count mismatch: {counts}")
    roles = list(EXPECTED_SPLIT_COUNTS)
    overlaps = [
        sorted(person_sets[roles[left]] & person_sets[roles[right]])
        for left in range(len(roles))
        for right in range(left + 1, len(roles))
    ]
    if any(overlaps):
        raise ValueError(f"person leakage across split roles: {overlaps}")


def load_ml_views(view_root: Path = ML_VIEW_ROOT) -> dict[str, pd.DataFrame]:
    manifest = verify_ml_view_manifest(view_root)
    views: dict[str, pd.DataFrame] = {}
    for split_role, metadata in manifest["files"].items():
        frame = pd.read_parquet(view_root / str(metadata["path"]))
        if len(frame) != metadata["row_count"]:
            raise ValueError(f"ML role view row count mismatch for {split_role}")
        missing = sorted(set(REQUIRED_VIEW_COLUMNS).difference(frame.columns))
        if missing:
            raise ValueError(f"ML role view required columns missing for {split_role}: {missing}")
        views[split_role] = _validate_public_evaluation_identity(frame.assign(split_role=split_role))
    _validate_loaded_split_contract(views)
    _validate_feature_contract_across_roles(views)
    return views


def _validated_feature_columns(
    frame: pd.DataFrame,
    feature_columns: Sequence[str],
) -> list[str]:
    columns = list(feature_columns)
    if not columns:
        raise ValueError("at least one feature column is required")
    missing = sorted(set(columns).difference(frame.columns))
    if missing:
        raise ValueError(f"feature columns missing: {missing}")
    unapproved = sorted(set(columns).difference(ALLOWED_FEATURE_COLUMNS))
    if unapproved:
        raise ValueError(f"unapproved columns requested as features: {unapproved}")
    nonnumeric = [
        column
        for column in columns
        if not (
            pd.api.types.is_numeric_dtype(frame[column].dtype)
            or pd.api.types.is_bool_dtype(frame[column].dtype)
        )
    ]
    if nonnumeric:
        raise ValueError(f"allowed feature columns must be numeric or bool: {sorted(nonnumeric)}")
    return columns


def _feature_matrix(frame: pd.DataFrame, feature_columns: Sequence[str]) -> np.ndarray:
    columns = _validated_feature_columns(frame, feature_columns)
    matrix = frame.loc[:, columns].to_numpy(dtype=np.float32)
    if not np.isfinite(matrix).all():
        raise ValueError("feature matrix contains non-finite values")
    return matrix


def _fit_binary_estimator(estimator: Any, matrix: np.ndarray, target: pd.Series, name: str) -> Any:
    values = target.to_numpy(dtype=np.int8)
    if set(np.unique(values)) != {0, 1}:
        raise ValueError(f"{name} needs both binary classes")
    estimator.fit(matrix, values)
    return estimator


def _select_stage_training_rows(frame: pd.DataFrame) -> pd.Series:
    return frame[PATTERN_TARGET].eq(1)


def _select_behavior_training_rows(frame: pd.DataFrame) -> pd.Series:
    hard_negative = frame["hard_negative"]
    if hard_negative.isna().any():
        raise ValueError("hard_negative contains null values")
    hard_negative_values = set(hard_negative.unique())
    if not hard_negative_values.issubset({0, 1, False, True}):
        raise ValueError("hard_negative must contain only exact binary values")
    hard_negative_rows = pd.Series(
        hard_negative.to_numpy(dtype=np.int8) == 1,
        index=frame.index,
        dtype=bool,
    )
    pattern_rows = pd.Series(
        frame[PATTERN_TARGET].to_numpy(dtype=np.int8) == 1,
        index=frame.index,
        dtype=bool,
    )
    behavior_positive = pd.Series(
        frame.loc[:, list(BEHAVIOR_CODES)]
        .eq(1)
        .any(axis=1)
        .to_numpy(dtype=bool),
        index=frame.index,
        dtype=bool,
    )
    invalid_positive = behavior_positive & ~(pattern_rows | hard_negative_rows)
    if invalid_positive.any():
        raise ValueError(
            "behavior-positive rows must be pattern or hard negative"
        )
    return pattern_rows | (hard_negative_rows & behavior_positive)


def _fit_multitask_candidate(
    frame: pd.DataFrame,
    feature_columns: Sequence[str],
    *,
    model_name: str,
    make_estimator: Any,
) -> dict[str, Any]:
    required_labels = {
        PATTERN_TARGET,
        ONSET_EVENT_TARGET,
        "hard_negative",
        STAGE_TARGET,
        *BEHAVIOR_CODES,
    }
    missing = sorted(required_labels.difference(frame.columns))
    if missing:
        raise ValueError(f"candidate training requires columns: {missing}")
    matrix = _feature_matrix(frame, feature_columns)
    pattern_model = _fit_binary_estimator(
        make_estimator(),
        matrix,
        frame[PATTERN_TARGET],
        f"{model_name} pattern",
    )

    stage_rows = frame.loc[_select_stage_training_rows(frame)].copy()
    if stage_rows.empty:
        raise ValueError(f"{model_name} needs pattern rows for conditional stage heads")
    stage_matrix = _feature_matrix(stage_rows, feature_columns)
    stage_models = {
        stage: _fit_binary_estimator(
            make_estimator(),
            stage_matrix,
            stage_rows[STAGE_TARGET].eq(stage),
            f"{model_name} stage {stage}",
        )
        for stage in STAGE_CODES
    }

    behavior_rows = frame.loc[_select_behavior_training_rows(frame)].copy()
    if behavior_rows.empty:
        raise ValueError(f"{model_name} needs behavior decision rows")
    behavior_matrix = _feature_matrix(behavior_rows, feature_columns)
    behavior_models = {
        behavior: _fit_binary_estimator(
            make_estimator(),
            behavior_matrix,
            behavior_rows[behavior],
            f"{model_name} behavior {behavior}",
        )
        for behavior in BEHAVIOR_CODES
    }
    return {
        "model_name": model_name,
        "pattern_model": pattern_model,
        "stage_models": stage_models,
        "behavior_models": behavior_models,
    }


def fit_logistic_candidate(
    frame: pd.DataFrame, feature_columns: Sequence[str]
) -> dict[str, Any]:
    def make_estimator() -> Pipeline:
        return Pipeline(
            [
                ("scale", StandardScaler()),
                ("model", LogisticRegression(
                    class_weight="balanced", max_iter=1000, random_state=17, solver="lbfgs"
                )),
            ]
        )

    return _fit_multitask_candidate(
        frame, feature_columns, model_name="logistic_regression", make_estimator=make_estimator
    )


def _make_hgb_estimator() -> HistGradientBoostingClassifier:
    return HistGradientBoostingClassifier(
        class_weight="balanced",
        max_iter=100,
        random_state=17,
    )


def fit_hgb_candidate(frame: pd.DataFrame, feature_columns: Sequence[str]) -> dict[str, Any]:
    return _fit_multitask_candidate(
        frame,
        feature_columns,
        model_name="hist_gradient_boosting",
        make_estimator=_make_hgb_estimator,
    )


## 2. Score validation only
Threshold and champion selection use validation rows exclusively. Locked-test metrics are reported only after validation has fixed the champion.

In [ ]:
def _positive_probability(model: Any, matrix: np.ndarray) -> np.ndarray:
    classes = np.asarray(model.classes_)
    positive_index = np.flatnonzero(classes == 1)
    if len(positive_index) != 1:
        raise ValueError("binary model does not expose a positive class")
    return model.predict_proba(matrix)[:, int(positive_index[0])].astype(np.float64)


def _segments(mask: np.ndarray) -> list[tuple[int, int]]:
    padded = np.pad(mask.astype(np.int8), (1, 1))
    changes = np.diff(padded)
    starts = np.flatnonzero(changes == 1)
    ends = np.flatnonzero(changes == -1) - 1
    return list(zip(starts.tolist(), ends.tolist(), strict=True))


def _event_alert_summary(
    truth: np.ndarray,
    predicted: np.ndarray,
    person_keys: np.ndarray,
    canonical_time: np.ndarray,
) -> tuple[float, float]:
    labels = np.asarray(truth, dtype=np.int8)
    alerts = np.asarray(predicted, dtype=bool)
    people = np.asarray(person_keys)
    times = np.asarray(canonical_time)
    lengths = {len(labels), len(alerts), len(people), len(times)}
    if len(lengths) != 1:
        raise ValueError("truth, predictions, person keys, and times must have equal lengths")
    grouped = pd.DataFrame({
        "truth": labels,
        "predicted": alerts,
        "person_key": people,
        "canonical_time": times,
    })
    detected = 0
    truth_event_count = 0
    false_alerts = 0
    for person_key, person in grouped.groupby("person_key", sort=False):
        if person["canonical_time"].duplicated().any():
            raise ValueError(f"duplicate canonical_time for person {person_key}")
        if not person["canonical_time"].is_monotonic_increasing:
            raise ValueError(f"canonical_time is not ordered for person {person_key}")
        person_truth = person["truth"].to_numpy(dtype=np.int8)
        person_predicted = person["predicted"].to_numpy(dtype=bool)
        truth_events = _segments(person_truth.astype(bool))
        truth_event_count += len(truth_events)
        detected += sum(
            bool(person_predicted[start : end + 1].any())
            for start, end in truth_events
        )
        false_alerts += len(
            _segments(person_predicted & ~person_truth.astype(bool))
        )
    event_recall = detected / truth_event_count if truth_event_count else 0.0
    return event_recall, float(false_alerts)


def expected_calibration_error(truth: np.ndarray, probability: np.ndarray, bins: int = 10) -> float:
    result = 0.0
    edges = np.linspace(0.0, 1.0, bins + 1)
    for lower, upper in zip(edges[:-1], edges[1:], strict=True):
        mask = (probability >= lower) & (probability < upper if upper < 1 else probability <= upper)
        if mask.any():
            result += float(mask.mean()) * abs(float(truth[mask].mean()) - float(probability[mask].mean()))
    return result


def select_validation_threshold(
    truth: np.ndarray,
    probability: np.ndarray,
    person_keys: np.ndarray,
    canonical_time: np.ndarray,
    *,
    duration_hours: float,
) -> float:
    lengths = {
        len(truth),
        len(probability),
        len(person_keys),
        len(canonical_time),
    }
    if len(lengths) != 1:
        raise ValueError("truth, probability, person keys, and times must have equal lengths")
    thresholds = np.unique(np.asarray(probability, dtype=np.float64))
    if not len(thresholds):
        return 0.5
    scores: list[tuple[float, float, float, float]] = []
    for threshold in thresholds:
        predicted = probability >= threshold
        event_recall, false_alerts = _event_alert_summary(
            truth,
            predicted,
            person_keys,
            canonical_time,
        )
        false_alerts_per_hour = false_alerts / duration_hours if duration_hours else 0.0
        scores.append((
            float(f1_score(truth, predicted, zero_division=0)),
            event_recall,
            -false_alerts_per_hour,
            float(threshold),
        ))
    return max(scores)[3]


def compute_common_metrics(
    truth: np.ndarray,
    probability: np.ndarray,
    *,
    threshold: float,
    duration_hours: float,
    model_name: str,
    split_role: str,
    target: str,
    person_keys: np.ndarray,
    canonical_time: np.ndarray,
) -> pd.DataFrame:
    labels = np.asarray(truth, dtype=np.int8)
    scores = np.asarray(probability, dtype=np.float64)
    if len(labels) != len(scores):
        raise ValueError("truth and probability lengths must match")
    if not len(labels):
        raise ValueError("metrics require at least one row")
    predicted = scores >= threshold
    event_recall, false_alerts = _event_alert_summary(
        labels,
        predicted,
        person_keys,
        canonical_time,
    )
    false_alerts_per_hour = false_alerts / duration_hours if duration_hours else 0.0
    aucpr = float(average_precision_score(labels, scores)) if len(np.unique(labels)) == 2 else np.nan
    auroc = float(roc_auc_score(labels, scores)) if len(np.unique(labels)) == 2 else np.nan
    metric_values = {
        "aucpr": aucpr,
        "auroc": auroc,
        "event_recall": event_recall,
        "row_recall": float(recall_score(labels, predicted, zero_division=0)),
        "row_f1": float(f1_score(labels, predicted, zero_division=0)),
        "false_alerts_per_hour": false_alerts_per_hour,
        "brier_score": float(brier_score_loss(labels, scores)),
        "ece": expected_calibration_error(labels, scores),
    }
    return pd.DataFrame(
        [
            {
                "model_family": "machine_learning",
                "model_name": model_name,
                "series_id": SERIES_ID,
                "split_role": split_role,
                "target": target,
                "metric": metric,
                "value": value,
                "support": int(labels.sum()),
                "data_status": DATA_STATUS,
            }
            for metric, value in metric_values.items()
        ],
        columns=METRIC_COLUMNS,
    )


def bootstrap_people_ci(
    frame: pd.DataFrame, *, iterations: int = 1000, random_state: int = 17
) -> dict[str, float]:
    required = {"person_key", "label", "probability"}
    missing = sorted(required.difference(frame.columns))
    if missing:
        raise ValueError(f"bootstrap rows missing columns: {missing}")
    people = np.sort(frame["person_key"].astype(str).unique())
    if len(people) < 2 or iterations < 1:
        raise ValueError("bootstrap requires at least two people and one iteration")

    def aucpr(rows: pd.DataFrame) -> float:
        labels = rows["label"].to_numpy(dtype=np.int8)
        scores = rows["probability"].to_numpy(dtype=np.float64)
        return float(average_precision_score(labels, scores)) if len(np.unique(labels)) == 2 else np.nan

    estimate = aucpr(frame)
    rng = np.random.default_rng(random_state)
    replicates: list[float] = []
    by_person = {person: frame.loc[frame["person_key"].astype(str).eq(person)] for person in people}
    for sample in rng.choice(people, size=(iterations, len(people)), replace=True):
        value = aucpr(pd.concat([by_person[person] for person in sample], ignore_index=True))
        if np.isfinite(value):
            replicates.append(value)
    if not replicates or not np.isfinite(estimate):
        raise ValueError("bootstrap samples have no two-class AUCPR")
    lower, upper = np.quantile(replicates, [0.025, 0.975])
    return {"estimate": estimate, "lower": float(lower), "upper": float(upper)}


def select_validation_champion(metric_rows: pd.DataFrame) -> str:
    validation = metric_rows.loc[
        metric_rows["split_role"].eq("validation")
        & metric_rows["target"].eq(PATTERN_TARGET)
    ].copy()
    required = {"aucpr", "event_recall", "false_alerts_per_hour", "ece"}
    table = validation.pivot_table(
        index="model_name", columns="metric", values="value", aggfunc="first"
    )
    missing = sorted(required.difference(table.columns))
    if missing:
        raise ValueError(f"validation champion metrics missing: {missing}")
    ranked = table.reset_index().sort_values(
        ["aucpr", "event_recall", "false_alerts_per_hour", "ece", "model_name"],
        ascending=[False, False, True, True, True],
        kind="mergesort",
    )
    if ranked.empty:
        raise ValueError("validation metrics are required for champion selection")
    return str(ranked.iloc[0]["model_name"])


## 3. Emit common-schema predictions and optional W&B logs
All modeling stays behind the explicit execution gate. The optional W&B login reads a Kaggle Secret at runtime only.

In [ ]:
def _validate_public_evaluation_identity(frame: pd.DataFrame) -> pd.DataFrame:
    required_text = ("person_key", "run_id", "dataset_id", "day_key", "session_id")
    missing = sorted(set((*required_text, "canonical_time")).difference(frame.columns))
    if missing:
        raise ValueError(f"public evaluation identity missing columns: {missing}")
    work = frame.copy()
    for column in required_text:
        invalid = work[column].isna() | ~work[column].map(
            lambda value: isinstance(value, str) and bool(value) and value == value.strip()
        )
        if invalid.any():
            raise ValueError(f"invalid public evaluation identity: {column}")
    if work["canonical_time"].isna().any():
        raise ValueError("canonical_time contains null or NaT")
    canonical_time = pd.to_datetime(work["canonical_time"], errors="raise")
    if not isinstance(canonical_time.dtype, pd.DatetimeTZDtype) or str(canonical_time.dtype.tz) != "UTC":
        raise ValueError("canonical_time must be UTC-aware")
    work["canonical_time"] = canonical_time
    return work


def _prediction_rows(
    candidate: dict[str, Any],
    frame: pd.DataFrame,
    feature_columns: Sequence[str],
    *,
    split_role: str,
    pattern_threshold: float,
) -> pd.DataFrame:
    frame = _validate_public_evaluation_identity(frame)
    rows: list[pd.DataFrame] = []

    def append_target(
        selected: pd.DataFrame,
        target: str,
        labels: np.ndarray,
        model: Any,
        threshold: float,
    ) -> None:
        matrix = _feature_matrix(selected, feature_columns)
        count = len(selected)
        rows.append(pd.DataFrame({
            "model_family": ["machine_learning"] * count,
            "model_name": [candidate["model_name"]] * count,
            "series_id": [SERIES_ID] * count,
            "dataset_id": selected["dataset_id"].to_numpy(),
            "run_id": selected["run_id"].to_numpy(),
            "person_key": selected["person_key"].to_numpy(),
            "day_key": selected["day_key"].to_numpy(),
            "session_id": selected["session_id"].to_numpy(),
            "canonical_time": selected["canonical_time"].to_numpy(),
            "split_role": [split_role] * count,
            "target": [target] * count,
            "label": labels.astype(np.int8),
            "probability": _positive_probability(model, matrix),
            "threshold": [threshold] * count,
        }, columns=PREDICTION_COLUMNS))

    append_target(
        frame,
        PATTERN_TARGET,
        frame[PATTERN_TARGET].to_numpy(),
        candidate["pattern_model"],
        pattern_threshold,
    )
    stage_frame = frame.loc[_select_stage_training_rows(frame)]
    for stage, model in candidate["stage_models"].items():
        append_target(
            stage_frame,
            f"stage::{stage}",
            stage_frame[STAGE_TARGET].eq(stage).to_numpy(),
            model,
            0.5,
        )
    behavior_frame = frame.loc[_select_behavior_training_rows(frame)]
    for behavior, model in candidate["behavior_models"].items():
        append_target(
            behavior_frame,
            f"behavior::{behavior}",
            behavior_frame[behavior].to_numpy(),
            model,
            0.5,
        )
    return pd.concat(rows, ignore_index=True)


def _infer_feature_columns(frame: pd.DataFrame) -> list[str]:
    unapproved = [
        column
        for column in frame.columns
        if column not in ALLOWED_FEATURE_COLUMNS
        and column not in EXACT_NON_FEATURE_COLUMNS
    ]
    if unapproved:
        raise ValueError(f"unapproved columns in ML role view: {sorted(unapproved)}")
    features = [
        column
        for column in ALLOWED_FEATURE_COLUMNS
        if column in frame.columns
    ]
    return _validated_feature_columns(frame, features)


def _validate_feature_contract_across_roles(
    views: dict[str, pd.DataFrame],
) -> list[str]:
    selected_features = _infer_feature_columns(views["train"])
    _feature_matrix(views["train"], selected_features)
    for split_role in ("validation", "locked_test"):
        role_features = _infer_feature_columns(views[split_role])
        if role_features != selected_features:
            raise ValueError(
                f"feature schema mismatch for {split_role}: "
                f"expected {selected_features}, got {role_features}"
            )
        _feature_matrix(views[split_role], selected_features)
    return selected_features


def _metrics_from_predictions(predictions: pd.DataFrame, *, duration_hours: float) -> pd.DataFrame:
    metrics = [
        compute_common_metrics(
            group["label"].to_numpy(),
            group["probability"].to_numpy(),
            threshold=float(group["threshold"].iloc[0]),
            duration_hours=duration_hours,
            model_name=str(group["model_name"].iloc[0]),
            split_role=str(group["split_role"].iloc[0]),
            target=str(target),
            person_keys=group["person_key"].to_numpy(),
            canonical_time=group["canonical_time"].to_numpy(),
        )
        for target, group in predictions.groupby("target", sort=True)
    ]
    return pd.concat(metrics, ignore_index=True)


def login_wandb_from_kaggle_secret() -> bool:
    try:
        from kaggle_secrets import UserSecretsClient
        import wandb
        key = UserSecretsClient().get_secret("WANDB_API_KEY")
    except Exception as exc:
        print(f"W&B 비활성화: {type(exc).__name__}")
        return False
    return bool(wandb.login(key=key, verify=True))


def run_ml_training() -> None:
    views = load_ml_views()
    train = views["train"]
    validation = views["validation"]
    feature_columns = _validate_feature_contract_across_roles(views)
    candidates = [
        fit_logistic_candidate(train, feature_columns),
        fit_hgb_candidate(train, feature_columns),
    ]
    validation_predictions: list[pd.DataFrame] = []
    validation_metrics: list[pd.DataFrame] = []
    duration_hours = len(validation) / 3600
    for candidate in candidates:
        pattern_probability = _positive_probability(
            candidate["pattern_model"], _feature_matrix(validation, feature_columns)
        )
        threshold = select_validation_threshold(
            validation[PATTERN_TARGET].to_numpy(),
            pattern_probability,
            validation["person_key"].to_numpy(),
            validation["canonical_time"].to_numpy(),
            duration_hours=duration_hours,
        )
        predictions = _prediction_rows(
            candidate, validation, feature_columns, split_role="validation", pattern_threshold=threshold
        )
        validation_predictions.append(predictions)
        validation_metrics.append(_metrics_from_predictions(predictions, duration_hours=duration_hours))
    all_validation_predictions = pd.concat(validation_predictions, ignore_index=True)
    all_validation_metrics = pd.concat(validation_metrics, ignore_index=True)
    champion_name = select_validation_champion(all_validation_metrics)
    ML_BENCHMARK_OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
    all_validation_predictions.to_parquet(
        ML_BENCHMARK_OUTPUT_ROOT / "validation_predictions.parquet", index=False, compression="zstd"
    )
    all_validation_metrics.to_parquet(
        ML_BENCHMARK_OUTPUT_ROOT / "validation_metrics.parquet", index=False, compression="zstd"
    )
    if RUN_LOCKED_TEST:
        champion = next(candidate for candidate in candidates if candidate["model_name"] == champion_name)
        threshold = float(all_validation_predictions.loc[
            (all_validation_predictions["model_name"] == champion_name)
            & (all_validation_predictions["target"] == PATTERN_TARGET),
            "threshold",
        ].iloc[0])
        locked = views["locked_test"]
        locked_predictions = _prediction_rows(
            champion, locked, feature_columns, split_role="locked_test", pattern_threshold=threshold
        )
        locked_predictions.to_parquet(
            ML_BENCHMARK_OUTPUT_ROOT / "locked_test_predictions.parquet", index=False, compression="zstd"
        )
        _metrics_from_predictions(locked_predictions, duration_hours=len(locked) / 3600).to_parquet(
            ML_BENCHMARK_OUTPUT_ROOT / "locked_test_metrics.parquet", index=False, compression="zstd"
        )
    if login_wandb_from_kaggle_secret():
        import wandb

        with wandb.init(
            project="multisensor-goal15-benchmark",
            group="machine-learning",
            tags=["oracle-sanity", "mvp3", "split-24-6-6", "not-real-verified"],
        ) as run:
            run.summary["validation_champion"] = champion_name
            wandb.log({"validation_metric_rows": len(all_validation_metrics)})


## 4. Explicit execution gate
The committed notebook does not validate data, train models, evaluate locked test, or contact W&B.

In [ ]:
if RUN_TRAINING:
    print("Round3 validation champion export 정의를 계속 로드합니다.")
else:
    print("학습 비활성화: RUN_TRAINING=False")


## 5. Round3 common-grid validation champion artifact
Only the selected validation champion is exported for one-to-one DL comparison. Threshold selection uses the same 1001-point event-level contract; locked-test rows are excluded.

In [ ]:
COMMON_THRESHOLD_GRID_SIZE = 101
CHAMPION_METRIC_COLUMNS = [*METRIC_COLUMNS, 'stress_condition']


def _common_grid_histogram(truth: np.ndarray, probability: np.ndarray) -> dict[str, Any]:
    labels = np.asarray(truth, dtype=np.int8)
    scores = np.asarray(probability, dtype=np.float64)
    bucket = np.minimum((scores * (COMMON_THRESHOLD_GRID_SIZE - 1)).astype(np.int64), COMMON_THRESHOLD_GRID_SIZE - 1)
    positive = np.zeros(COMMON_THRESHOLD_GRID_SIZE)
    negative = np.zeros(COMMON_THRESHOLD_GRID_SIZE)
    probability_sum = np.zeros(COMMON_THRESHOLD_GRID_SIZE)
    np.add.at(positive, bucket[labels == 1], 1)
    np.add.at(negative, bucket[labels == 0], 1)
    np.add.at(probability_sum, bucket, scores)
    return {
        'positive': positive, 'negative': negative, 'probability_sum': probability_sum,
        'brier_sum': float(np.square(scores - labels).sum()),
    }


def _common_grid_binary_metrics(
    truth: np.ndarray, probability: np.ndarray, *, threshold: float,
) -> dict[str, float]:
    state = _common_grid_histogram(truth, probability)
    positive = state['positive']
    negative = state['negative']
    total_positive = float(positive.sum())
    total_negative = float(negative.sum())
    total = total_positive + total_negative
    tp = np.cumsum(positive[::-1])
    fp = np.cumsum(negative[::-1])
    recall = np.divide(tp, total_positive, out=np.zeros_like(tp), where=total_positive > 0)
    precision = np.divide(tp, tp + fp, out=np.ones_like(tp), where=(tp + fp) > 0)
    global_aucpr = float(np.sum(np.diff(np.r_[0.0, recall]) * precision)) if total_positive and total_negative else np.nan
    fpr = np.divide(fp, total_negative, out=np.zeros_like(fp), where=total_negative > 0)
    global_auroc = float(np.trapezoid(np.r_[0.0, recall], np.r_[0.0, fpr])) if total_positive and total_negative else np.nan
    threshold_index = int(round(threshold * (COMMON_THRESHOLD_GRID_SIZE - 1)))
    reverse_index = COMMON_THRESHOLD_GRID_SIZE - 1 - threshold_index
    selected_tp = float(tp[reverse_index])
    selected_fp = float(fp[reverse_index])
    selected_fn = total_positive - selected_tp
    denominator = 2 * selected_tp + selected_fp + selected_fn
    count = positive + negative
    calibration = np.divide(state['probability_sum'], count, out=np.zeros_like(count), where=count > 0)
    observed = np.divide(positive, count, out=np.zeros_like(count), where=count > 0)
    return {
        'global_aucpr': global_aucpr, 'global_auroc': global_auroc,
        'global_row_f1': 2 * selected_tp / denominator if denominator else 0.0,
        'global_brier_score': float(state['brier_sum'] / total) if total else np.nan,
        'global_ece': float(np.sum(count * np.abs(observed - calibration)) / total) if total else np.nan,
    }


def _internal_metric_segment_identity(frame: pd.DataFrame) -> pd.DataFrame:
    '''Internal array-metric adapter; public frame entrypoints validate before this path.'''
    work = frame.copy()
    if 'canonical_time' not in work or work['canonical_time'].isna().any():
        raise ValueError('internal metric canonical_time is required')
    if pd.api.types.is_numeric_dtype(work['canonical_time']):
        work['canonical_time'] = pd.to_datetime(work['canonical_time'], unit='s', utc=True)
    else:
        canonical_time = pd.to_datetime(work['canonical_time'], errors='raise')
        if not isinstance(canonical_time.dtype, pd.DatetimeTZDtype) or str(canonical_time.dtype.tz) != 'UTC':
            raise ValueError('internal metric canonical_time must be UTC-aware')
        work['canonical_time'] = canonical_time
    if 'person_key' not in work or work['person_key'].isna().any():
        raise ValueError('internal metric person_key is required')
    for column in ('run_id', 'dataset_id'):
        if column not in work:
            work[column] = '__internal_array_metric__'
    if 'day_key' not in work:
        work['day_key'] = work['canonical_time'].dt.strftime('%Y-%m-%d')
    if 'session_id' not in work:
        work['session_id'] = '__internal_array_metric__'
    return work


def _common_grid_event_statistics(frame: pd.DataFrame) -> dict[str, Any]:
    frame = _internal_metric_segment_identity(frame)
    detected = np.zeros(COMMON_THRESHOLD_GRID_SIZE)
    false_alerts = np.zeros(COMMON_THRESHOLD_GRID_SIZE)
    truth_events = 0
    duration_hours = 0.0
    for _, person in frame.groupby('person_key', sort=True):
        ordered = person.sort_values('canonical_time', kind='mergesort')
        truth = ordered['label'].to_numpy(dtype=np.int8).astype(bool)
        probability = ordered['probability'].to_numpy(dtype=np.float64)
        bucket = np.minimum((probability * (COMMON_THRESHOLD_GRID_SIZE - 1)).astype(np.int64), COMMON_THRESHOLD_GRID_SIZE - 1)
        detected_difference = np.zeros(COMMON_THRESHOLD_GRID_SIZE)
        false_difference = np.zeros(COMMON_THRESHOLD_GRID_SIZE)
        segments = _segments(truth)
        truth_events += len(segments)
        for start, end in segments:
            detected_difference[0] += 1
            maximum = int(bucket[start:end + 1].max())
            if maximum + 1 < COMMON_THRESHOLD_GRID_SIZE:
                detected_difference[maximum + 1] -= 1
        for index in np.flatnonzero(~truth):
            current = int(bucket[index])
            lower = 0 if index == 0 or truth[index - 1] else int(bucket[index - 1]) + 1
            if lower <= current:
                false_difference[lower] += 1
                if current + 1 < COMMON_THRESHOLD_GRID_SIZE:
                    false_difference[current + 1] -= 1
        detected += np.cumsum(detected_difference)
        false_alerts += np.cumsum(false_difference)
        times = pd.to_datetime(ordered['canonical_time'], utc=True)
        duration_hours += max(float((times.iloc[-1] - times.iloc[0]).total_seconds()) / 3600, 1 / 3600)
    return {
        'detected': detected, 'false_alerts': false_alerts,
        'truth_events': truth_events, 'duration_hours': duration_hours,
    }


def select_validation_threshold(
    truth: np.ndarray, probability: np.ndarray, person_keys: np.ndarray,
    canonical_time: np.ndarray, *, duration_hours: float,
) -> float:
    frame = pd.DataFrame({
        'label': truth, 'probability': probability,
        'person_key': person_keys, 'canonical_time': canonical_time,
    })
    event = _common_grid_event_statistics(frame)
    detected = event['detected']
    false_alerts = event['false_alerts']
    truth_events = int(event['truth_events'])
    event_recall = np.divide(detected, truth_events, out=np.zeros_like(detected), where=truth_events > 0)
    event_level_f1 = np.divide(
        2 * detected, truth_events + detected + false_alerts,
        out=np.zeros_like(detected), where=(truth_events + detected + false_alerts) > 0,
    )
    false_alerts_per_hour = false_alerts / max(float(event['duration_hours']), 1 / 3600)
    grid = np.linspace(0.0, 1.0, COMMON_THRESHOLD_GRID_SIZE)
    best = max(
        range(COMMON_THRESHOLD_GRID_SIZE),
        key=lambda index: (
            float(event_level_f1[index]), float(event_recall[index]),
            -float(false_alerts_per_hour[index]), float(grid[index]),
        ),
    )
    _ = duration_hours
    return float(grid[best])


def compute_champion_grid_metrics(predictions: pd.DataFrame) -> pd.DataFrame:
    if not predictions['split_role'].eq('validation').all():
        raise ValueError('champion common-grid metrics accept validation only')
    rows: list[dict[str, Any]] = []
    for target, group in predictions.groupby('target', sort=True):
        threshold = float(group['threshold'].iloc[0])
        values = _common_grid_binary_metrics(group['label'].to_numpy(), group['probability'].to_numpy(), threshold=threshold)
        if target == PATTERN_TARGET:
            event = _common_grid_event_statistics(group)
            index = int(round(threshold * (COMMON_THRESHOLD_GRID_SIZE - 1)))
            detected = float(event['detected'][index])
            false_alerts = float(event['false_alerts'][index])
            truth_events = int(event['truth_events'])
            values.update({
                'global_event_recall': detected / truth_events if truth_events else 0.0,
                'global_false_alerts_per_hour': false_alerts / max(float(event['duration_hours']), 1 / 3600),
            })
        for metric, value in values.items():
            rows.append({
                'model_family': 'machine_learning', 'model_name': str(group['model_name'].iloc[0]),
                'series_id': SERIES_ID, 'split_role': 'validation', 'target': str(target),
                'metric': metric, 'value': value, 'support': len(group),
                'data_status': DATA_STATUS, 'stress_condition': 'clean',
            })
    return pd.DataFrame(rows, columns=CHAMPION_METRIC_COLUMNS)


def select_validation_champion(metric_rows: pd.DataFrame) -> str:
    validation = metric_rows.loc[metric_rows['split_role'].eq('validation') & metric_rows['target'].eq(PATTERN_TARGET)]
    table = validation.pivot(index='model_name', columns='metric', values='value')
    required = {'global_aucpr', 'global_event_recall', 'global_false_alerts_per_hour', 'global_ece'}
    legacy_required = {'aucpr', 'event_recall', 'false_alerts_per_hour', 'ece'}
    if not required.issubset(table.columns):
        if not legacy_required.issubset(table.columns):
            raise ValueError('validation champion common-grid metrics are incomplete')
        required = legacy_required
        table = table.rename(columns={
            'aucpr': 'global_aucpr', 'event_recall': 'global_event_recall',
            'false_alerts_per_hour': 'global_false_alerts_per_hour',
            'ece': 'global_ece',
        })
    ranked = table.reset_index().sort_values(
        ['global_aucpr', 'global_event_recall', 'global_false_alerts_per_hour', 'global_ece', 'model_name'],
        ascending=[False, False, True, True, True], kind='mergesort',
    )
    return str(ranked.iloc[0]['model_name'])


def write_validation_champion_artifact(
    metrics: pd.DataFrame, predictions: pd.DataFrame, *, champion_name: str,
    feature_columns: Sequence[str], view_manifest: Mapping[str, Any],
) -> tuple[Path, Path]:
    selected = metrics.loc[
        metrics['model_name'].eq(champion_name)
        & metrics['split_role'].eq('validation')
        & metrics['target'].eq(PATTERN_TARGET)
    ].copy()
    keys = ['target', 'metric']
    if selected.empty or selected.duplicated(keys).any() or selected['stress_condition'].ne('clean').any():
        raise ValueError('validation champion metrics must have one unique (target, metric) grain')
    threshold = float(predictions.loc[
        predictions['model_name'].eq(champion_name) & predictions['target'].eq(PATTERN_TARGET),
        'threshold',
    ].iloc[0])
    output_path = ML_BENCHMARK_OUTPUT_ROOT / 'validation_champion_metrics.parquet'
    manifest_path = ML_BENCHMARK_OUTPUT_ROOT / 'validation_champion_metrics.manifest.json'
    selected.to_parquet(output_path, index=False, compression='zstd')
    label_payload = json.dumps({'pattern': PATTERN_TARGET, 'onset_audit': ONSET_EVENT_TARGET, 'stages': STAGE_CODES, 'behaviors': BEHAVIOR_CODES}, sort_keys=True)
    feature_payload = json.dumps(list(feature_columns), separators=(',', ':'))
    manifest_path.write_text(json.dumps({
        'schema_version': 'goal1.5/ml-validation-champion/v1',
        'source_dataset_hash': view_manifest['source_dataset_hash'],
        'split_hash': view_manifest['split_hash'],
        'label_schema_hash': hashlib.sha256(label_payload.encode()).hexdigest(),
        'feature_schema_hash': hashlib.sha256(feature_payload.encode()).hexdigest(),
        'threshold': threshold, 'grid_size': COMMON_THRESHOLD_GRID_SIZE,
        'model_family': 'machine_learning', 'model_name': champion_name,
        'target': PATTERN_TARGET, 'split_role': 'validation',
        'file': output_path.name, 'file_sha256': sha256_file(output_path),
    }, indent=2, sort_keys=True) + '\n')
    return output_path, manifest_path


In [ ]:
def run_ml_training() -> None:
    view_manifest = verify_ml_view_manifest()
    views = load_ml_views()
    train, validation = views['train'], views['validation']
    feature_columns = _validate_feature_contract_across_roles(views)
    candidates = [fit_logistic_candidate(train, feature_columns), fit_hgb_candidate(train, feature_columns)]
    validation_predictions: list[pd.DataFrame] = []
    validation_metrics: list[pd.DataFrame] = []
    duration_hours = len(validation) / 3600
    for candidate in candidates:
        probability = _positive_probability(candidate['pattern_model'], _feature_matrix(validation, feature_columns))
        threshold = select_validation_threshold(
            validation[PATTERN_TARGET].to_numpy(), probability,
            validation['person_key'].to_numpy(), validation['canonical_time'].to_numpy(),
            duration_hours=duration_hours,
        )
        predictions = _prediction_rows(candidate, validation, feature_columns, split_role='validation', pattern_threshold=threshold)
        validation_predictions.append(predictions)
        validation_metrics.append(compute_champion_grid_metrics(predictions))
    all_predictions = pd.concat(validation_predictions, ignore_index=True)
    all_metrics = pd.concat(validation_metrics, ignore_index=True)
    champion_name = select_validation_champion(all_metrics)
    ML_BENCHMARK_OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
    all_predictions.to_parquet(ML_BENCHMARK_OUTPUT_ROOT / 'validation_predictions.parquet', index=False, compression='zstd')
    all_metrics.to_parquet(ML_BENCHMARK_OUTPUT_ROOT / 'validation_metrics.parquet', index=False, compression='zstd')
    write_validation_champion_artifact(
        all_metrics, all_predictions, champion_name=champion_name,
        feature_columns=feature_columns, view_manifest=view_manifest,
    )
    if RUN_LOCKED_TEST:
        raise RuntimeError('locked test requires a separate audited invocation')
    if login_wandb_from_kaggle_secret():
        import wandb
        with wandb.init(project='multisensor-goal15-benchmark', group='machine-learning') as run:
            run.summary['validation_champion'] = champion_name


if RUN_TRAINING:
    print('Round4 champion prediction export 정의를 계속 로드합니다.')
else:
    print('학습 비활성화: validation champion artifact를 생성하지 않았습니다.')


In [ ]:
def _prediction_endpoint_coverage_hash(predictions: pd.DataFrame) -> str:
    endpoints = predictions.loc[predictions['target'].eq(PATTERN_TARGET), [
        'dataset_id', 'run_id', 'person_key', 'canonical_time',
    ]].sort_values(['person_key', 'run_id', 'dataset_id', 'canonical_time'], kind='mergesort')
    if endpoints.empty or endpoints.duplicated().any():
        raise ValueError('champion endpoint identity must be non-empty and unique')
    digest = hashlib.sha256()
    for row in endpoints.itertuples(index=False, name=None):
        digest.update(('\x1f'.join(map(str, row)) + '\n').encode())
    return digest.hexdigest()


def write_validation_champion_artifact(
    metrics: pd.DataFrame, predictions: pd.DataFrame, *, champion_name: str,
    feature_columns: Sequence[str], view_manifest: Mapping[str, Any],
) -> tuple[Path, Path]:
    selected_predictions = predictions.loc[
        predictions['model_name'].eq(champion_name)
        & predictions['split_role'].eq('validation')
    ].copy()
    required_targets = {PATTERN_TARGET, *{f'stage::{code}' for code in STAGE_CODES}, *{f'behavior::{code}' for code in BEHAVIOR_CODES}}
    if set(selected_predictions['target']) != required_targets:
        raise ValueError('validation champion predictions must cover every hierarchical target')
    selected_predictions['target_model_id'] = selected_predictions['target'].map(
        lambda target: f'{champion_name}::{target}'
    )
    target_order = {target: index for index, target in enumerate([
        PATTERN_TARGET, *[f'stage::{code}' for code in STAGE_CODES],
        *[f'behavior::{code}' for code in BEHAVIOR_CODES],
    ])}
    selected_predictions['_target_order'] = selected_predictions['target'].map(target_order)
    selected_predictions = selected_predictions.sort_values([
        'person_key', 'run_id', 'dataset_id', 'canonical_time', '_target_order',
    ], kind='mergesort').drop(columns='_target_order').reset_index(drop=True)
    endpoint_coverage_hash = _prediction_endpoint_coverage_hash(selected_predictions)
    selected_metrics = metrics.loc[
        metrics['model_name'].eq(champion_name)
        & metrics['split_role'].eq('validation')
        & metrics['stress_condition'].eq('clean')
    ].copy()
    if selected_metrics.empty or selected_metrics.duplicated(['target', 'metric']).any():
        raise ValueError('validation champion metrics must have a unique target/metric grain')
    ML_BENCHMARK_OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
    prediction_path = ML_BENCHMARK_OUTPUT_ROOT / 'validation_champion_predictions.parquet'
    metrics_path = ML_BENCHMARK_OUTPUT_ROOT / 'validation_champion_metrics.parquet'
    manifest_path = ML_BENCHMARK_OUTPUT_ROOT / 'validation_champion_predictions.manifest.json'
    selected_predictions.to_parquet(prediction_path, index=False, compression='zstd')
    selected_metrics.to_parquet(metrics_path, index=False, compression='zstd')
    label_payload = json.dumps({'pattern': PATTERN_TARGET, 'onset_audit': ONSET_EVENT_TARGET, 'stages': STAGE_CODES, 'behaviors': BEHAVIOR_CODES}, sort_keys=True)
    feature_payload = json.dumps(list(feature_columns), separators=(',', ':'))
    manifest_path.write_text(json.dumps({
        'schema_version': 'goal1.5/ml-validation-champion-predictions/v1',
        'source_dataset_hash': view_manifest['source_dataset_hash'],
        'split_hash': view_manifest['split_hash'],
        'label_schema_hash': hashlib.sha256(label_payload.encode()).hexdigest(),
        'feature_schema_hash': hashlib.sha256(feature_payload.encode()).hexdigest(),
        'endpoint_coverage_hash': endpoint_coverage_hash,
        'prediction_sha256': sha256_file(prediction_path),
        'metrics_sha256': sha256_file(metrics_path),
        'model_family': 'machine_learning', 'model_name': champion_name,
        'target_model_ids': sorted(selected_predictions['target_model_id'].unique()),
        'targets': sorted(required_targets), 'split_role': 'validation',
        'prediction_file': prediction_path.name, 'metrics_file': metrics_path.name,
    }, indent=2, sort_keys=True) + '\n')
    return prediction_path, manifest_path


def run_ml_training() -> None:
    view_manifest = verify_ml_view_manifest()
    views = load_ml_views()
    train, validation = views['train'], views['validation']
    feature_columns = _validate_feature_contract_across_roles(views)
    candidates = [fit_logistic_candidate(train, feature_columns), fit_hgb_candidate(train, feature_columns)]
    prediction_frames: list[pd.DataFrame] = []
    metric_frames: list[pd.DataFrame] = []
    for candidate in candidates:
        probability = _positive_probability(candidate['pattern_model'], _feature_matrix(validation, feature_columns))
        threshold = select_validation_threshold(
            validation[PATTERN_TARGET].to_numpy(), probability,
            validation['person_key'].to_numpy(), validation['canonical_time'].to_numpy(),
            duration_hours=len(validation) / 3600,
        )
        candidate_predictions = _prediction_rows(
            candidate, validation, feature_columns,
            split_role='validation', pattern_threshold=threshold,
        )
        prediction_frames.append(candidate_predictions)
        metric_frames.append(compute_champion_grid_metrics(candidate_predictions))
    all_predictions = pd.concat(prediction_frames, ignore_index=True)
    all_metrics = pd.concat(metric_frames, ignore_index=True)
    champion_name = select_validation_champion(all_metrics)
    write_validation_champion_artifact(
        all_metrics, all_predictions, champion_name=champion_name,
        feature_columns=feature_columns, view_manifest=view_manifest,
    )
    if RUN_LOCKED_TEST:
        raise RuntimeError('locked test requires a separate audited invocation')


if RUN_TRAINING:
    print('Round5 공통 event-run 정의를 계속 로드합니다.')
else:
    print('학습 비활성화: ML champion prediction artifact를 생성하지 않았습니다.')


In [ ]:
def _common_grid_event_statistics(frame: pd.DataFrame) -> dict[str, Any]:
    detected = np.zeros(COMMON_THRESHOLD_GRID_SIZE)
    false_alerts = np.zeros(COMMON_THRESHOLD_GRID_SIZE)
    truth_events = 0
    duration_hours = 0.0
    grid = np.linspace(0.0, 1.0, COMMON_THRESHOLD_GRID_SIZE)
    for _, person in frame.groupby('person_key', sort=True):
        ordered = person.sort_values('canonical_time', kind='mergesort')
        truth = ordered['label'].to_numpy(dtype=np.int8).astype(bool)
        probability = ordered['probability'].to_numpy(dtype=np.float64)
        buckets = np.minimum((probability * (COMMON_THRESHOLD_GRID_SIZE - 1)).astype(np.int64), COMMON_THRESHOLD_GRID_SIZE - 1)
        alert_active = np.zeros(COMMON_THRESHOLD_GRID_SIZE, dtype=bool)
        alert_truth_overlap = np.zeros(COMMON_THRESHOLD_GRID_SIZE, dtype=bool)
        truth_active = False
        truth_event_max_bucket = -1
        for label, score, bucket in zip(truth, probability, buckets, strict=True):
            if label:
                if not truth_active:
                    truth_active = True
                    truth_event_max_bucket = int(bucket)
                else:
                    truth_event_max_bucket = max(truth_event_max_bucket, int(bucket))
            elif truth_active:
                truth_events += 1
                detected[:truth_event_max_bucket + 1] += 1
                truth_active = False
                truth_event_max_bucket = -1
            active = score >= grid
            ending = alert_active & ~active
            false_alerts += ending & ~alert_truth_overlap
            starting = active & ~alert_active
            alert_truth_overlap[starting] = bool(label)
            continuing = active & alert_active
            if label:
                alert_truth_overlap[continuing] = True
            alert_truth_overlap[~active] = False
            alert_active = active
        if truth_active:
            truth_events += 1
            detected[:truth_event_max_bucket + 1] += 1
        false_alerts += alert_active & ~alert_truth_overlap
        times = pd.to_datetime(ordered['canonical_time'], utc=True)
        duration_hours += max(float((times.iloc[-1] - times.iloc[0]).total_seconds()) / 3600, 1 / 3600)
    return {
        'detected': detected, 'false_alerts': false_alerts,
        'truth_events': truth_events, 'duration_hours': duration_hours,
    }


if RUN_TRAINING:
    print('Round6 segment boundary 정의를 계속 로드합니다.')
else:
    print('학습 비활성화: 공통 maximal alert-run event 평가를 실행하지 않았습니다.')


In [ ]:
class SegmentEventGridState:
    '''Event grid whose exposure is observed 1 Hz rows and whose state never crosses a segment boundary.'''
    SEGMENT_FIELDS = ('person_key', 'run_id', 'dataset_id', 'day_key', 'session_id')

    def __init__(self, *, bins: int = COMMON_THRESHOLD_GRID_SIZE) -> None:
        if bins < 2:
            raise ValueError('event grid requires at least two bins')
        self.bins = bins
        self.grid = np.linspace(0.0, 1.0, bins)
        self.detected = np.zeros(bins, dtype=np.float64)
        self.false_alerts = np.zeros(bins, dtype=np.float64)
        self.truth_events = 0
        self.observed_rows = 0
        self.last_identity: tuple[Any, ...] | None = None
        self.last_timestamp: pd.Timestamp | None = None
        self.truth_active = False
        self.truth_event_max_bucket = -1
        self.alert_active = np.zeros(bins, dtype=bool)
        self.alert_truth_overlap = np.zeros(bins, dtype=bool)

    def _identity(self, row: Mapping[str, Any]) -> tuple[Any, ...]:
        values: list[str] = []
        for field in self.SEGMENT_FIELDS:
            if field not in row or pd.isna(row[field]):
                raise ValueError(f'missing segment identity: {field}')
            value = row[field]
            if not isinstance(value, str) or not value or value != value.strip():
                raise ValueError(f'invalid segment identity: {field}')
            values.append(value)
        return tuple(values)

    def _close_truth_event(self) -> None:
        if self.truth_active:
            self.truth_events += 1
            self.detected[:self.truth_event_max_bucket + 1] += 1
            self.truth_active = False
            self.truth_event_max_bucket = -1

    def _close_segment(self) -> None:
        self._close_truth_event()
        self.false_alerts += self.alert_active & ~self.alert_truth_overlap
        self.alert_active.fill(False)
        self.alert_truth_overlap.fill(False)

    def update_row(
        self, row: Mapping[str, Any], *, truth: int, probability: float,
    ) -> None:
        if 'canonical_time' not in row or pd.isna(row['canonical_time']):
            raise ValueError('canonical_time is required')
        timestamp = pd.Timestamp(row['canonical_time'])
        if timestamp.tzinfo is None or str(timestamp.tz) != 'UTC':
            raise ValueError('canonical_time must be UTC-aware')
        identity = self._identity(row)
        boundary = (
            self.last_identity is None
            or identity != self.last_identity
            or timestamp - self.last_timestamp != pd.Timedelta(seconds=1)
        )
        if boundary and self.last_identity is not None:
            self._close_segment()
        score = float(probability)
        if score < 0.0 or score > 1.0:
            raise ValueError('event probability must be in [0, 1]')
        label = bool(truth)
        bucket = min(int(score * (self.bins - 1)), self.bins - 1)
        if label:
            if not self.truth_active:
                self.truth_active = True
                self.truth_event_max_bucket = bucket
            else:
                self.truth_event_max_bucket = max(self.truth_event_max_bucket, bucket)
        else:
            self._close_truth_event()
        active = score >= self.grid
        ending = self.alert_active & ~active
        self.false_alerts += ending & ~self.alert_truth_overlap
        starting = active & ~self.alert_active
        self.alert_truth_overlap[starting] = label
        if label:
            self.alert_truth_overlap[active & self.alert_active] = True
        self.alert_truth_overlap[~active] = False
        self.alert_active = active
        self.observed_rows += 1
        self.last_identity = identity
        self.last_timestamp = timestamp

    def finalize(self) -> dict[str, Any]:
        self._close_segment()
        return {
            'detected': self.detected.copy(),
            'false_alerts': self.false_alerts.copy(),
            'truth_events': int(self.truth_events),
            'duration_hours': self.observed_rows / 3600,
        }


def _common_grid_event_statistics(frame: pd.DataFrame) -> dict[str, Any]:
    if not {'run_id', 'dataset_id', 'day_key', 'session_id'}.issubset(frame.columns):
        frame = _internal_metric_segment_identity(frame)
    state = SegmentEventGridState(bins=COMMON_THRESHOLD_GRID_SIZE)
    sort_columns = [column for column in ('person_key', 'run_id', 'dataset_id', 'canonical_time') if column in frame]
    ordered = frame.sort_values(sort_columns, kind='mergesort')
    for _, row in ordered.iterrows():
        state.update_row(
            row, truth=int(row['label']), probability=float(row['probability']),
        )
    result = state.finalize()
    if result['duration_hours'] != state.observed_rows / 3600:
        raise AssertionError('event exposure must equal observed_rows / 3600')
    return result


if RUN_TRAINING:
    run_ml_training()
else:
    print('학습 비활성화: segment-aware ML event 평가를 실행하지 않았습니다.')


## Round 6 세그먼트 경계 계약

사람·run·dataset·선택적 day/session 변경 또는 정확히 1초가 아닌 시간 간격에서 사건과 예측 alert 상태를 닫습니다. 노출 시간은 벽시계 간격이 아니라 실제로 관찰된 1 Hz 행 수만 사용합니다.